

## โจทย์การบ้าน: การพัฒนา Two-Agent System

### วัตถุประสงค์

ให้นักศึกษาออกแบบและสร้างระบบ Agent จำนวน 2 ตัวเพื่อทำงานร่วมกัน โดยมีเป้าหมายเพื่อตอบคำถามที่ซับซ้อนได้อย่างถูกต้องและมีคุณภาพสูง

1.  **Agent 1 (ReAct Agent)**: มีหน้าที่หลักในการรวบรวมข้อมูลและคำนวณ โดยใช้เทคนิค ReAct (Reasoning and Acting) ในการตัดสินใจว่าจะใช้เครื่องมือ (Tools) ใดเพื่อหาข้อมูลที่จำเป็น Agent 1 ควรจะสามารถ:

      * **ค้นหาข้อมูล**: ดึงข้อมูลจากฐานความรู้ (เช่น Wikipedia) หรือเว็บเพจที่กำหนด
      * **คำนวณ**: ใช้เครื่องมือคำนวณเพื่อแก้ปัญหาทางคณิตศาสตร์
      * **สังเคราะห์คำตอบ**: รวบรวมข้อมูลดิบที่ได้จากเครื่องมือและสร้างคำตอบเบื้องต้น

2.  **Agent 2 (Self-Reflecting Agent)**: มีหน้าที่ประเมินและปรับปรุงคำตอบที่ได้จาก Agent 1 Agent 2 ควรจะสามารถ:

      * **วิเคราะห์คำตอบ**: ตรวจสอบคำตอบเบื้องต้นที่สร้างโดย Agent 1 เพื่อหาข้อผิดพลาด ความไม่สมบูรณ์ หรือจุดที่สามารถปรับปรุงได้
      * **สร้าง Feedback**: สร้าง feedback ที่เฉพาะเจาะจงเพื่อชี้แนะแนวทางในการแก้ไขคำตอบ
      * **ปรับปรุงคำตอบ**: ใช้ feedback ที่สร้างขึ้นเพื่อปรับปรุงคำตอบให้ถูกต้องและสมบูรณ์ยิ่งขึ้น

### คำถามสำหรับทดสอบ (จำนวน 10 ข้อ)

ให้นำ Agent ทั้งสองมาทดสอบด้วยชุดคำถามต่อไปนี้ โดยคำถามเหล่านี้ถูกออกแบบมาให้ต้องใช้ทั้งการค้นหาข้อมูล การคำนวณ และการสังเคราะห์ข้อมูลที่ซับซ้อน

```python
questions = [
    "ผลคูณของ 147 กับ 258 คือเท่าไหร่?",
    "สรุปประวัติของ AI จากหน้า Wikipedia เป็นภาษาไทย",
    "Elon Musk ก่อตั้งบริษัทใดบ้าง และปัจจุบันมีตำแหน่งอะไรในแต่ละบริษัท?",
    "ค้นหาและสรุปข้อมูลล่าสุดเกี่ยวกับ iPhone รุ่นใหม่",
    "คำนวณค่าของ (15 + 7) * 3 / 2",
    "ข้อมูลเศรษฐกิจของประเทศไทย ณ ปี 2024 เป็นอย่างไรบ้าง? (ให้อ้างอิงแหล่งที่มา)",
    "ดาวเคราะห์ดวงใดในระบบสุริยะที่มีขนาดใหญ่ที่สุดและมีจำนวนดวงจันทร์เท่าใด?",
    "สี่เหลี่ยมจัตุรัสที่มีพื้นที่ 256 ตารางเมตร จะมีความยาวเส้นรอบรูปเท่าใด?",
    "อธิบายแนวคิดหลักของ 'การเรียนรู้ของเครื่อง' (Machine Learning) พร้อมยกตัวอย่าง 2 ตัวอย่าง",
    "เปรียบเทียบข้อดีและข้อเสียของภาษา Python และ Java"
]
```


In [17]:
import os
from openai import OpenAI
from dotenv import load_dotenv
import IPython
import sys
import json

def clean_notebook():
    IPython.display.clear_output(wait=True)
    print("Notebook cleaned.")

!pip install openai wikipedia sympy requests -q

clean_notebook()
# Load environment variables
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
model_name =  "gpt-4o"   


Notebook cleaned.


In [5]:
from sympy import sympify
import wikipedia

def wikipedia_lookup(query: str) -> str:
    """Searches Wikipedia for matching page titles.
    
    Args:
        query: The search query string.
        
    Returns:
        A JSON string of Wikipedia page titles.
    """
    try:
        return json.dumps(wikipedia.search(query))
    except Exception as e:
        return f"Error: {e}"

def get_wikipedia_page(title: str) -> str:
    """Gets a snippet of content from a specific Wikipedia page.
    
    Args:
        title: The exact title of the Wikipedia page.
        
    Returns:
        A string containing the first 2000 characters of the page content.
    """
    try:
        page = wikipedia.page(title, auto_suggest=False)
        return page.content[:2000]
    except wikipedia.exceptions.PageError:
        return "Page not found"
    except wikipedia.exceptions.DisambiguationError as e:
        return f"Disambiguation page. Possible options: {e.options}"
    except Exception as e:
        return f"Error: {e}"

def calculator(expression: str) -> str:
    """Useful for mathematical calculations.
    
    Args:
        expression: A mathematical expression string (e.g., "2+3*4").
    
    Returns:
        The result of the calculation as a string.
    """
    try:
        return str(sympify(expression))
    except Exception as e:
        return f"Error: {e}"

In [9]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Useful for mathematical calculations. Input: a mathematical expression like '2+3*4'.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "The mathematical expression to evaluate"}
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "wikipedia_lookup",
            "description": "Searches Wikipedia for page titles. Input: a query.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The search query."}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_wikipedia_page",
            "description": "Gets content from a Wikipedia page. Input: the exact page title.",
            "parameters": {
                "type": "object",
                "properties": {
                    "title": {"type": "string", "description": "The exact page title."}
                },
                "required": ["title"]
            }
        }
    }, 
]

available_functions = {
    "calculator": calculator,
    "wikipedia_lookup": wikipedia_lookup,
    "get_wikipedia_page": get_wikipedia_page,
}

In [15]:
def llm_generator(messages):
    """
    Generate a response from the LLM based on the input messages.
    
    Args:
    messages (list): List of message dictionaries, e.g., [{"role": "system", "content": "..."}, {"role": "user", "content": "..."}]
    
    Returns:
    str: The generated response content.
    """
    response = client.chat.completions.create(
        model=model_name,
        messages=messages,
        temperature=0.1,  # Adjust temperature for creativity

    )
    return response.choices[0].message.content

def reflect_agent(answer,question):
    """
    Analyze the given answer using the reflective prompt and return the feedback.
    
    Args:
    answer (str): The answer to reflect on.
    
    Returns:
    str: The reflection feedback as a string.
    """
    reflect_prompt = (
        f"คุณเป็นผู้เชี่ยวชาญในการประเมินคำตอบ กรุณาวิเคราะห์คำตอบต่อไปนี้ in thai:\n"
        f"- 'completeness': ประเมินความครบถ้วนของคำตอบ (คะแนน 1-10)\n"
        f"- 'accuracy': ประเมินความถูกต้องของคำตอบ (คะแนน 1-10)\n"
        f"- 'clarity': ประเมินความชัดเจนของคำตอบ (คะแนน 1-10)\n"
        f"- 'strengths': จุดแข็งของคำตอบ\n"
        f"- 'weaknesses': จุดอย่อนของคำตอบ\n"
        f"- 'missing_aspects': สิ่งที่ขาดหายไปในคำตอบ\n"
        f"- 'improvement_suggestions': ข้อเสนอแนะเพื่อปรับปรุงคำตอบ\n\n"
        f"คำถาม: {question}\n"
        f"คำตอบ: {answer}\n"
    )
    
    # Messages for the reflect function
    messages = [
        {"role": "system", "content": reflect_prompt}
    ]
    
    # Call the LLM to generate the reflection (using the same generator function for simplicity)
    feedback = llm_generator(messages)
    return feedback

def run_agent(query, max_steps=10):
    messages = [
        {"role": "system", "content": "You are a helpful assistant that uses tools to answer questions."},
        {"role": "user", "content": query}
    ]
    
    print(f"Starting agent for query: {query}\n")
    
    for step in range(max_steps):
        print(f"--- Step {step + 1} ---")
        
        response = client.chat.completions.create(
            model=model_name,
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )
        
        response_message = response.choices[0].message
        messages.append(response_message)
        
        tool_calls = response_message.tool_calls
        
        if tool_calls:
            print(f"Agent wants to use tools. Tool calls: {tool_calls}\n")
            
            for tool_call in tool_calls:
                function_name = tool_call.function.name
                function_to_call = available_functions[function_name]
                function_args = json.loads(tool_call.function.arguments)
                
                print(f"Executing tool: {function_name} with arguments: {function_args}")
                
                # Check if the tool expects arguments, if not, call without
                if function_args:
                    function_response = function_to_call(**function_args)
                else:
                    function_response = function_to_call()
                
                print(f"Observation: {function_response}\n")
                
                messages.append(
                    {
                        "tool_call_id": tool_call.id,
                        "role": "tool",
                        "name": function_name,
                        "content": function_response,
                    }
                )
        else:
            final_answer = response_message.content
            print(f"Final Answer: {final_answer}\n")
            return final_answer
    
    print("Agent failed to reach a final answer.")
    return "Agent failed to reach a final answer."

In [19]:
questions = [
    "ผลคูณของ 147 กับ 258 คือเท่าไหร่?",
    "สรุปประวัติของ AI จากหน้า Wikipedia เป็นภาษาไทย",
    "Elon Musk ก่อตั้งบริษัทใดบ้าง และปัจจุบันมีตำแหน่งอะไรในแต่ละบริษัท?",
    "ค้นหาและสรุปข้อมูลล่าสุดเกี่ยวกับ iPhone รุ่นใหม่",
    "คำนวณค่าของ (15 + 7) * 3 / 2",
    "ข้อมูลเศรษฐกิจของประเทศไทย ณ ปี 2024 เป็นอย่างไรบ้าง? (ให้อ้างอิงแหล่งที่มา)",
    "ดาวเคราะห์ดวงใดในระบบสุริยะที่มีขนาดใหญ่ที่สุดและมีจำนวนดวงจันทร์เท่าใด?",
    "สี่เหลี่ยมจัตุรัสที่มีพื้นที่ 256 ตารางเมตร จะมีความยาวเส้นรอบรูปเท่าใด?",
    "อธิบายแนวคิดหลักของ 'การเรียนรู้ของเครื่อง' (Machine Learning) พร้อมยกตัวอย่าง 2 ตัวอย่าง",
    "เปรียบเทียบข้อดีและข้อเสียของภาษา Python และ Java"
]

for i, q in enumerate(questions, 1):
    print(f"Question {i}: {q}")
    answer = run_agent(q)
    print(f"Answer: {answer}\n")
    reflect_result = reflect_agent(answer,q)
    print(f"ผลการวิเคราะห์คำตอบ: \n{reflect_result}\n")
    print("-" * 50)
    print("")

Question 1: สรุปประวัติของ AI จากหน้า Wikipedia เป็นภาษาไทย
Starting agent for query: สรุปประวัติของ AI จากหน้า Wikipedia เป็นภาษาไทย

--- Step 1 ---
Agent wants to use tools. Tool calls: [ChatCompletionMessageToolCall(id='call_T1WYnaXlJxI4hQZSjZwt7M7X', function=Function(arguments='{"query":"AI history"}', name='wikipedia_lookup'), type='function')]

Executing tool: wikipedia_lookup with arguments: {'query': 'AI history'}
Observation: ["History of artificial intelligence", "OpenAI", "AI boom", "Perplexity AI", "AI winter", "Artificial intelligence", "Symbolic artificial intelligence", "Empire of AI", "Artificial intelligence visual art", "Generative artificial intelligence"]

--- Step 2 ---
Agent wants to use tools. Tool calls: [ChatCompletionMessageToolCall(id='call_B6xghODNTK6JUal9g0i3i0Ez', function=Function(arguments='{"title":"History of artificial intelligence"}', name='get_wikipedia_page'), type='function')]

Executing tool: get_wikipedia_page with arguments: {'title': 'History